In [ ]:

import tsim
import stim
import pyzx as zx
import pyzx_param as param

In [ ]:
circ = tsim.Circuit(
"""
CX 0 1
CX 1 0
X 1
"""
)

In [ ]:
for x in circ:
    print(circ)
    g = circ.pop(index=0)
    x = g.targets_copy()
    circ.append_from_stim_program_text(g.name + " " + str(x[0].qubit_value) + " " + str(x[1].qubit_value))


In [ ]:
for t in range(len(circ)):
    print(1)

In [ ]:
len(circ)

In [ ]:
g = circ.pop(index=0)
print(g.tag)
print(g.name)
x = g.targets_copy()
x

In [ ]:
y = [0,0]

for i in range(0, len(x), 2):
                print(x[i].qubit_value)
                print(x[i + 1].qubit_value)

In [ ]:
z0 = 1
z1 = 0

#calculate probability
denom = abs(z0) ** 2 + abs(z1) ** 2
p = abs(z0) ** 2 / denom

p

In [ ]:
import numpy as np

In [ ]:
had_mat = np.array([[1, 1], [1, -1]], dtype=complex) / np.sqrt(2)

In [ ]:
def _bitstring_to_index(bits: list[int]) -> int:
    """Convert a bitstring [b_0, b_1, ..., b_{n-1}] to an integer index."""
    idx = 0
    for b in bits:
        idx = (idx << 1) | b
    return idx


In [ ]:
n_qubits = 2
state = np.zeros(2 ** n_qubits, dtype=complex)

state[0] = 1
print(state)

target = 0
state = state.reshape(2, n_qubits)
print(state)

# Move target axis to front for easy contraction
state = np.moveaxis(state, target, 0)
state = np.einsum("ij,j...->i...", had_mat, state)
state = np.moveaxis(state, 0, target)
print(state)
state = state.reshape(-1)
print(state)


In [ ]:
x_index = _bitstring_to_index([0,0])
complex(state[x_index])


In [ ]:
def gate_by_gate(circuit: tsim.Circuit):
    circ_final = "I"
    y = [0] * circuit.num_qubits
    for t in range(len(circuit)):
        gate = circ.pop(index=0)
        # Gate is a CNOT so update the output classically
        if gate.name == ("CX" or "CNOT" or "ZCX"):
            targets = gate.targets_copy()
            for i in range(0, len(targets), 2):
                a = targets[i].qubit_value
                b = targets[i + 1].qubit_value
                if y[a] == 1:
                    y[b] = 1 - y[b]
        if gate.name == "H":
    return y

In [ ]:
gate_by_gate(circ)


In [ ]:

import tsim
import stim
import pyzx as zx
import gate_by_gate as gbg
import tsim
import numpy as np
from fractions import Fraction
import pyzx_param as param

In [ ]:

x = [0, 0]

test_circ = tsim.Circuit("""

    H 0
    CX 0 1
    """)
num_qubits = test_circ.num_qubits
# test_circ.append_from_stim_program_text(f"R {' '.join(str(q) for q in range(num_qubits))}")
g = test_circ.diagram("pyzx")

# For each qubit, find the vertex with the highest row number
last_vertices = {}
for v in g.vertices():
    q = g.qubit(v)
    if q not in last_vertices or g.row(v) > g.row(last_vertices[q]):
        last_vertices[q] = v

print(last_vertices)
zx.draw(g, labels=True)

In [ ]:
for qubit, bit in enumerate(x):
    out_vertex = last_vertices[qubit]
    phase = Fraction(0) if bit == 0 else Fraction(1)  # 0 = |0>, pi = |1>
    # Insert a Z-spider with the right phase before the output
    g.set_type(out_vertex, zx.VertexType.Z)
    g.set_phase(out_vertex, phase)

In [ ]:

zx.draw(g, labels=True)

In [ ]:
param.full_reduce(g, paramSafe=True)

In [ ]:
complex(g.scalar.to_number())

In [ ]:
num_qubits = 4
circ_until_now = tsim.Circuit()

circ_until_now.append_from_stim_program_text(f"R {' '.join(str(q) for q in range(num_qubits))}")
circ_until_now.append_from_stim_program_text(f"R {' '.join(str(q) for q in range(num_qubits))}")

circ_until_now

In [ ]:
g = circ_until_now.diagram("pyzx")


In [ ]:
zx.full_reduce(g)
zx.draw(g)

In [ ]:
g.outputs()

In [1]:
import tsim
import stim
import pyzx as zx
import gate_by_gate as gbg
import tsim
import numpy as np
from fractions import Fraction
import pyzx_param as param

In [2]:
print("=== Bell-state circuit ===")
# Produces |Φ+> = (|00> + |11>) / sqrt(2)
# Expected: samples should be 00 or 11 with equal probability

bell_circuit = tsim.Circuit(
"""
H 0
CX 0 1
"""
)

# (|000> + |111>) / sqrt(2) — should only see 000 or 111
ghz_circuit = tsim.Circuit(
    """
    H 0
    CX 0 1 0 2
    """
)

x_circuit = tsim.Circuit(
    """
    X 0
    CX 0 1
    """
)

example_circuit = tsim.Circuit(
    """
    H 0
    CX 0 1
    H 1
    CX 0 1
    H 0
    """
)

reset_circuit = tsim.Circuit(
    """
    X 0 1 2 3
    CX 0 1
    R 0
    """
)

resetX_circuit = tsim.Circuit(
    """
    RX 0
    """
)

check_circuit = tsim.Circuit(
    """
    X 0
    H 0
    H 0
    """
)

measure_circ = tsim.Circuit("""
    RX 0
    MX 0 1
""")

detector_circ = tsim.Circuit(
    """
    H 0
    M 0
    DETECTOR rec[-1]
    """
)

rng = np.random.default_rng(42)
counts = {"00": 0, "11": 0, "01": 0, "10": 0}
N = 100
detects = None
for _ in range(N):
    passed, result, detects = gbg.gate_by_gate(x_circuit.copy(), detects)
    if passed:
        key = "".join(map(str, result))
        counts[key] = counts.get(key, 0) + 1

print(f"Samples from {N} runs:")
for k, v in sorted(counts.items()):
    if v > 0:
        print(f"  |{k}>: {v} ({100*v/N:.1f}%)")


=== Bell-state circuit ===
Samples from 100 runs:
  |11>: 100 (100.0%)


In [51]:
preproc_circ = tsim.Circuit(
    """
    """
)

In [106]:
gbg.preprocessing(check_circuit)

[Graph(0 vertices, 0 edges), Graph(0 vertices, 0 edges)]

In [39]:
bell_circuit = tsim.Circuit(
"""
R 0
H 0
"""
)

In [89]:
uggy = param.Graph()

In [90]:
uggy.add_vertex(ty = zx.VertexType.X)

0

In [91]:
uggy.add_vertex(ty = zx.VertexType.Z)
uggy.add_edge([0,1], zx.EdgeType.HADAMARD)

[0, 1]

In [92]:
uggy.add_vertex(ty = zx.VertexType.X)
uggy.add_edge([1,2], zx.EdgeType.SIMPLE)

[1, 2]

In [93]:
uggy.add_params(2, {'a'})

In [94]:
uggy.add_vertex(ty=zx.VertexType.X)
uggy.add_edge([1, 3], zx.EdgeType.SIMPLE)


[1, 3]

In [95]:
uggy.add_params(3, {'b'})

In [97]:
uggy.add_vertex(ty=zx.VertexType.Z)
uggy.add_edge([4, 3], zx.EdgeType.HADAMARD)


[4, 3]

In [98]:
param.draw(uggy, labels=True)

In [99]:
param.full_reduce(uggy)
param.draw(uggy, labels=True)

In [103]:
print(g.scalar.phasevars_halfpi)
print(g.scalar.phasevars_pi)
print(g.scalar.phasevars_pi_pair)
print(g.scalar.phasenodevars)


{}
set()
[]
[]


In [105]:
uggy.scalar

Scalar(2.00+0.00i = sqrt(2)^0(1+exp(0ipi)))

In [21]:
g = bell_circuit.diagram("pyzx")


In [104]:
zx.draw(g, labels=True)

In [88]:
g.scalar

Scalar(0.71+0.00i = sqrt(2)^-1)

In [48]:
bell_circuit = tsim.Circuit(
"""
R 0 1 2
"""
)

In [16]:
bell_circuit.append_from_stim_program_text(f"T 0")

In [50]:
g = bell_circuit.diagram("pyzx")
zx.draw(g, labels=True)

In [42]:
gate = bell_circuit.pop()
targets = gate.targets_copy()
gate.name

'T'

In [44]:
gate.name


'S'

In [43]:
gate.tag == 'T'

True

In [54]:
g.outputs

<bound method GraphS.outputs of Graph(7 vertices, 4 edges)>

In [20]:
last_vertices = {}
for v in g.vertices():
    if v.ty == zx.VertexType.BOUNDARY:
        last_vertices[q] = v

AttributeError: 'int' object has no attribute 'ty'

In [19]:
print(last_vertices)
x = [0]

NameError: name 'last_vertices' is not defined

In [33]:
for qubit, bit in enumerate(x):
    out_vertex = last_vertices[qubit]
    # phase = Fraction(0) if bit == 0 else Fraction(1)  # 0 = |0>, pi = |1>
    # Insert a Z-spider with the right phase before the output
    g.set_type(out_vertex, zx.VertexType.X)
    if qubit == 0: g.add_params(out_vertex, 'a')
    else: g.add_params(out_vertex, 'b')

TypeError: 'int' object is not iterable

In [31]:
param.draw(g, labels=True)

In [32]:
for x in g.vertices():
    print(x, g.get_params(x))


0 set()
1 set()
2 set()


In [230]:
param.full_reduce(g, paramSafe=True)
param.draw(g, labels=True)

In [231]:
g.scalar.phasevars_halfpi
g.scalar.phasevars_pi
g.scalar.phasevars_pi_pair
g.scalar.phasenodevars



{}

In [232]:
g.scalar.phasevars_pi_pair

[]

In [233]:
g.scalar.phasevars_pi

set()

In [234]:
g.scalar.phasenodevars

[{'a'}]

In [237]:
'a' in g.scalar.phasenodevars[0]

True

In [ ]:
00, 01, 10 , 11

In [ ]:
zx.draw(g, labels=True)

In [ ]:
param.full_reduce(g, paramSafe=True)

In [ ]:
amplitude = complex(g.scalar.to_number())
a = amplitude

In [ ]:
a

In [ ]:
amplitude = complex(g.scalar.to_number())
a = amplitude

In [ ]:
x = [0]

In [ ]:
for qubit, bit in enumerate(x):
    out_vertex = last_vertices[qubit]
    phase = Fraction(0) if bit == 0 else Fraction(1)  # 0 = |0>, pi = |1>
    # Insert a Z-spider with the right phase before the output
    g.set_type(out_vertex, zx.VertexType.X)
    g.set_phase(out_vertex, phase)

In [ ]:
bell_circuit = tsim.Circuit(
"""
X_ERROR(0.1) 0
"""
)

In [ ]:
g = bell_circuit.pop(index=0)
g.gate_args_copy()

In [ ]:
zx.draw(g, labels=True)

In [ ]:
for gate in bell_circuit:
    targets = gate.targets_copy()
    print(targets)

In [ ]:
bell_circuit = tsim.Circuit(
"""
R 0
H 0
"""
)

In [14]:
import string
import itertools

def generate_labels(n, start_label: string = None):
    labels = []
    length = 1
    while len(labels) < n:
        for combo in itertools.product(string.ascii_lowercase, repeat=length):
            m = ''.join(combo)
            if start_label is not None:
                m = start_label + m
            labels.append(m)
            if len(labels) == n:
                break
        length += 1
    return labels

In [17]:
lbls = generate_labels(30, "n_")
lbls

['n_a',
 'n_b',
 'n_c',
 'n_d',
 'n_e',
 'n_f',
 'n_g',
 'n_h',
 'n_i',
 'n_j',
 'n_k',
 'n_l',
 'n_m',
 'n_n',
 'n_o',
 'n_p',
 'n_q',
 'n_r',
 'n_s',
 'n_t',
 'n_u',
 'n_v',
 'n_w',
 'n_x',
 'n_y',
 'n_z',
 'n_aa',
 'n_ab',
 'n_ac',
 'n_ad']